In [1]:
"""
A Gradio demo web application developed for Cardiac MRI segmentation.
"""

# Import libraries
import os
import tempfile
import shutil
import gradio as gr
import torch
import numpy as np
import albumentations as A
from pathlib import Path
from PIL import Image
from glob import glob
from functools import partial
from src.org_unet import UNet
from src.residual_unet import ResidualUNet
from src.attention_unet import AttentionUNetV3
from src.feature_pyramid_unet import FeaturePyramidUNet
from src.feedback_resunet import FeedbackResUNet
from src.transUnet import TransformerUNet

# Constants
LV = "Left Ventricle"
RV = "Right Ventricle"
MYO = "Myocardium"

# Setup device
device = torch.device('cpu')
current_dir = Path.cwd()

# Setup class configs
IMG_CSS="""
#image {
      display: block;
      margin-left: auto;
      margin-right: auto;
}
"""
CLASSES: tuple = (LV, RV, MYO)
CLASS_DESCRIPTIONS = {
    LV: "The Left Ventricle is the chamber of the heart that pumps oxygenated blood to the body via the aorta.",
    RV: "The Right Ventricle pumps deoxygenated blood to the lungs through the pulmonary artery.",
    MYO: "The Myocardium is the muscular middle layer of the heart wall responsible for contracting and pumping blood."
}
class2hexcolor = {LV: "#FF0000", RV: "#007fff", MYO: "#009A17"}
class_colors = {
    0: (0, 0, 0),        # Background - Black
    1: (0, 0, 255),      # LV - Red
    2: (0, 255, 0),      # RV - Blue
    3: (255, 0, 0),      # MYO - Green
}

# Randomly pick example images
images_dir = glob(os.path.join("../data/ACDC/img_slices_with_ratios_v4/testing/labeled_-1/", "images") + os.sep + "*.png")
examples = [i for i in np.random.choice(images_dir, size=8, replace=False)]


def delete_directory(req: gr.Request):
    if not req.username:
        return
    user_dir: Path = current_dir / req.username
    shutil.rmtree(str(user_dir))


def load_model(checkpoint_path, model_class):
    """
    Load a model from a given checkpoint.

    Parameters:
        checkpoint_path (Path): Path to the model checkpoint.
        model_class: Class of the model architecture.

    Returns:
        torch.nn.Module: Loaded model.
    """
    model_ = model_class()
    checkpoint = torch.load(checkpoint_path)
    model_.load_state_dict(checkpoint)
    model_ = model_.eval()
    return model_


def preprocess_image(image):
    """
    Preprocess the image for segmentation.
    Parameters:
        image (np.array): Image to be preprocessed.
    Returns:
        image (np.array): Preprocessed image.
    """
    image = np.array(image.resize((224, 224))).astype(np.float32) / 255.0  # [0,1]
    if image.ndim == 3:
        image = image[..., 0]  # Convert RGB to grayscale if needed

    image = (image * 255).astype(np.uint8)  # Scale to [0, 255]

    # Normalize using Albumentations
    normalize_transform = A.Normalize(mean=0.5, std=0.5, max_pixel_value=1.0)
    normalized = normalize_transform(image=image)
    image = torch.from_numpy(normalized["image"]).unsqueeze(0).unsqueeze(0)  #Add batch and channel dimensions [1, 1, H, W]

    return image.float()


def apply_colormap(mask):
    """Convert class mask to RGB image."""
    h, w = mask.shape
    rgb_mask = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in class_colors.items():
        rgb_mask[mask == cls] = color
    return Image.fromarray(rgb_mask)


def get_class_description(evt: gr.SelectData):
    """
    Function to get class description.
    :param evt: Select data.
    :return: Class description
    """
    return CLASS_DESCRIPTIONS.get(CLASSES[evt.index], "Select a region to see details.")


def segment_cmri(image):
    """
    Segment the image using a trained model.
    Parameters:
        image (np.array): Image to be segmented.
    Returns:
        results (np.array): Segmented image.
    """
    # Load model
    model_path = Path(f'./models/checkpoints/feat_pyramid_unet_img_slices_with_ratios_v4_20241224021156-rumbling-dog-720/checkpoint_epoch{23}.pth')
    model = load_model(checkpoint_path=model_path, model_class=FeaturePyramidUNet)

    # Preprocess the image
    org_img = image.copy()
    image = preprocess_image(image)

    with torch.no_grad():
        image = image.to(device, dtype=torch.float32, memory_format=torch.channels_last)

        model = model.to(device)

        prediction = model(image)  # Get model prediction
        prediction = torch.argmax(prediction, dim=1).squeeze(0).cpu().numpy()

        seg_info = [(prediction == idx, class_name) for idx, class_name in enumerate(CLASSES, 1)]

        # To visualize the segmentation mask only
        # colored_mask = apply_colormap(prediction)

    return org_img, seg_info


C:\Users\Chinthalanka\anaconda3\envs\pytorch_gpu\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.5 (you have 1.4.7). Upgrade using: pip install --upgrade albumentations


In [2]:
# Gradio Demo - v1.0
'''
demo = gr.Interface(
    fn=segment_cmri,
    inputs=gr.Image(type="pil", image_mode='L', height=360, width=360, label="Input Image", elem_id="image",),
    outputs=gr.AnnotatedImage(label="Segmented Image", elem_id="image", height=360, width=360, color_map=class2hexcolor),
    css=IMG_CSS,
    title="Cardiac MRI Segmentation",
    examples=examples
)
'''

'\ndemo = gr.Interface(\n    fn=segment_cmri,\n    inputs=gr.Image(type="pil", image_mode=\'L\', height=360, width=360, label="Input Image", elem_id="image",),\n    outputs=gr.AnnotatedImage(label="Segmented Image", elem_id="image", height=360, width=360, color_map=class2hexcolor),\n    css=IMG_CSS,\n    title="Cardiac MRI Segmentation",\n    examples=examples\n)\n'

In [3]:
# Gradio Demo - v1.1
'''
with gr.Blocks(title="Cardiac MRI Segmentation", theme=gr.themes.Soft()) as demo:
        gr.Markdown("""<h1><center>Cardiac MRI Segmentation with ACDC Dataset</center></h1>""")

        with gr.Row():
            img_input = gr.Image(type="pil", height=360, width=360, label="Input Image")
            img_output = gr.AnnotatedImage(label="Segmented Image", height=360, width=360, color_map=class2hexcolor)

        with gr.Row():
            predict_btn = gr.Button(value="Submit")
            clear_btn = gr.ClearButton(value="Clear", components=[img_input, img_output])

        predict_btn.click(
            fn=partial(segment_cmri),
            inputs=img_input,
            outputs=img_output
        )

        gr.Examples(examples=examples, inputs=img_input, outputs=img_output)
'''

'\nwith gr.Blocks(title="Cardiac MRI Segmentation", theme=gr.themes.Soft()) as demo:\n        gr.Markdown("""<h1><center>Cardiac MRI Segmentation with ACDC Dataset</center></h1>""")\n\n        with gr.Row():\n            img_input = gr.Image(type="pil", height=360, width=360, label="Input Image")\n            img_output = gr.AnnotatedImage(label="Segmented Image", height=360, width=360, color_map=class2hexcolor)\n\n        with gr.Row():\n            predict_btn = gr.Button(value="Submit")\n            clear_btn = gr.ClearButton(value="Clear", components=[img_input, img_output])\n\n        predict_btn.click(\n            fn=partial(segment_cmri),\n            inputs=img_input,\n            outputs=img_output\n        )\n\n        gr.Examples(examples=examples, inputs=img_input, outputs=img_output)\n'

In [4]:
# Gradio Demo - v1.2
'''
def create_overlay_and_save(annotated):
    """
    Create orange overlay and return image path
    :param annotated: Annotated image
    :return:
    """
    # Save the annotated image to a temporary file
    tmp_dir = tempfile.mkdtemp()
    path = os.path.join(tmp_dir, "annotated_segmentation.png")
    annotated.save(path)
    return path

with gr.Blocks(
    title="Cardiac MRI Segmentation",
    css="""
    #predict-btn > .gr-button {
        background-color: orange !important;
        color: white !important;
    }
    """
) as demo:

    gr.Markdown("<h1><center>Cardiac MRI Segmentation with ACDC Dataset</center></h1>")

    with gr.Row():
        img_input = gr.Image(type="pil", height=360, width=360, label="Input Image")
        img_output = gr.AnnotatedImage(
            label="Segmented Image",
            height=360,
            width=360,
            color_map=class2hexcolor
        )

    with gr.Row():
        predict_btn = gr.Button("Generate Predictions", elem_id="predict-btn")
        download_btn = gr.DownloadButton(label="Download Annotated Image")
        clear_btn = gr.ClearButton(components=[img_input, img_output], value="Clear")

    # Prediction: input image ➜ output (image, annotation)
    predict_btn.click(
        fn=segment_cmri,
        inputs=img_input,
        outputs=img_output
    )

    # Download the overlaid image
    download_btn.click(
        fn=create_overlay_and_save,
        inputs=img_output,
        outputs=download_btn
    )

    # Example image gallery
    gr.Examples(
        examples=examples,
        inputs=img_input,
        outputs=img_output,
        fn=segment_cmri,
        cache_examples=False
    )
'''

'\ndef create_overlay_and_save(annotated):\n    """\n    Create orange overlay and return image path\n    :param annotated: Annotated image\n    :return:\n    """\n    # Save the annotated image to a temporary file\n    tmp_dir = tempfile.mkdtemp()\n    path = os.path.join(tmp_dir, "annotated_segmentation.png")\n    annotated.save(path)\n    return path\n\nwith gr.Blocks(\n    title="Cardiac MRI Segmentation",\n    css="""\n    #predict-btn > .gr-button {\n        background-color: orange !important;\n        color: white !important;\n    }\n    """\n) as demo:\n\n    gr.Markdown("<h1><center>Cardiac MRI Segmentation with ACDC Dataset</center></h1>")\n\n    with gr.Row():\n        img_input = gr.Image(type="pil", height=360, width=360, label="Input Image")\n        img_output = gr.AnnotatedImage(\n            label="Segmented Image",\n            height=360,\n            width=360,\n            color_map=class2hexcolor\n        )\n\n    with gr.Row():\n        predict_btn = gr.Button("

In [5]:
# Gradio Demo - v1.2
def get_downloadable_annotated_image(output):
    """
    Generates a downloadable segmented image by overlaying class-specific colored masks
    on top of the original input image.

    Parameters:
        output (tuple): A tuple of the form (image_path, annotations), where:
                        - image_path (str): Path to the original input image.
                        - annotations (list): A list of tuples in the form (mask_path, label),
                                              where each mask_path is the path to a binary
                                              mask image and label is the class name.

    Returns:
        str: Path to the temporary image file (.png) containing the overlaid segmentation masks.
             This path can be directly used with gr.DownloadButton in Gradio.

    Notes:
        - Overlay colors are determined using the `class2hexcolor` dictionary.
        - The overlay is semi-transparent for better visualization on top of the original image.
        - The image is saved as a temporary file and is not automatically deleted.
    """
    if not output or not isinstance(output, tuple):
        return None

    image_path, annotations = output
    image = Image.open(image_path).convert("RGBA")
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))

    for mask_path, label in annotations:
        color_hex = class2hexcolor.get(label, "#FFFFFF")
        r, g, b = tuple(int(color_hex.lstrip("#")[i:i+2], 16) for i in (0, 2, 4))

        mask_img = Image.open(mask_path).convert("L").resize(image.size)
        color_mask = Image.new("RGBA", image.size, (r, g, b, 255)) # Adjusted opacity of the mask from the last parameter
        overlay.paste(color_mask, (0, 0), mask_img)

    blended = Image.alpha_composite(image, overlay)

    # Save to a real temp file instead of buffer
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    blended.save(temp_file.name, format="PNG")
    temp_file.close()

    return temp_file.name


with gr.Blocks(title="Cardiac MRI Segmentation", theme=gr.themes.Soft(), delete_cache=(86400, 86400)) as demo:
    gr.Markdown("""<h1><center>Cardiac MRI Segmentation with ACDC Dataset</center></h1>""")

    with gr.Row():
        img_input = gr.Image(type="pil", height=360, width=360, label="Input Image")
        img_output = gr.AnnotatedImage(label="Segmented Image", height=360, width=360, color_map=class2hexcolor)

    with gr.Row():
        predict_btn = gr.Button(value="Submit")
        download_btn = gr.DownloadButton(label="Download Output")

    with gr.Row():
        class_info_box = gr.Textbox(
            label="Region Description",
            # value="Select a region in the Segmented Image to see details.",
            # placeholder="Select a region in the Segmented Image to see details.",
            info="Select a region in the Segmented Image to see details.",
            interactive=False,
            max_lines=4
        )

    with gr.Row():
        clear_btn = gr.ClearButton(value="Clear", components=[img_input, img_output, class_info_box])

    predict_btn.click(
        fn=partial(segment_cmri),
        inputs=img_input,
        outputs=img_output
    )

    download_btn.click(
        fn=get_downloadable_annotated_image,
        inputs=img_output,
        outputs=download_btn
    )

    img_output.select(
        fn=get_class_description,
        inputs=None,
        outputs=class_info_box
    )

    gr.Examples(examples=examples, inputs=img_input, outputs=img_output)

    demo.unload(delete_directory)

In [6]:
demo.launch(share=False, debug=True)

INFO:httpx:HTTP Request: GET https://checkip.amazonaws.com/ "HTTP/1.1 200 "


Running on local URL:  http://127.0.0.1:7860


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"



To create a public link, set `share=True` in `launch()`.


INFO:httpx:HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


Keyboard interruption in main thread... closing server.
